In [1]:
# 1) Install once in your environment (run in terminal, not in notebook):
# pip install chembl_webresource_client pandas

from chembl_webresource_client.new_client import new_client
import pandas as pd
# ✅ Your folder path
save_path = r"C:\Users\kiran\Cheminformatics_course\New_Project\Absorption\Data_set"
# -------- SETTINGS YOU CAN EDIT --------
# Standard types that correspond to HIA / permeability assays
ABS_TYPES = [
    "PCaco-2", "Papp", "Papp e-6", "Permeability", "Permeability coefficient",
    "Absorption", "Pcaco2", "Cell permeation", "Papps", "Permeability rate",
    "permeability", "logPapp", "LogPapp", "Papp A to B (mean)",
    "Caco-2 A-B", "Caco-2 Papp", "Caco-2 permeability", "Drugabsorption"
]

# Max number of records PER type (ChEMBL can be very big)
MAX_RECORDS_PER_TYPE = 30000

# Output file
OUT_CSV = "chembl_hia_raw_from_api.csv"
# ---------------------------------------


def fetch_absorption_from_chembl():
    activities = new_client.activity

    all_rows = []

    for stype in ABS_TYPES:
        print(f"Fetching type: {stype}")
        # Basic filter: only numeric values with a standard value
        q = activities.filter(standard_type=stype, standard_value__isnull=False)

        # Restrict columns to only what we need (keeps CSV light)
        q = q.only([
            "molecule_chembl_id",
            "assay_chembl_id",
            "standard_type",
            "standard_units",
            "standard_value",
            "canonical_smiles"
        ])

        # Pull records (limit to avoid HUGE downloads)
        rows = []
        for i, rec in enumerate(q):
            if i >= MAX_RECORDS_PER_TYPE:
                break
            rows.append(rec)

        print(f"  → got {len(rows)} records")
        all_rows.extend(rows)

    print(f"\nTotal raw rows collected: {len(all_rows)}")

    df = pd.DataFrame(all_rows)
    print("Initial DataFrame shape:", df.shape)

    # Basic cleaning: drop rows without SMILES or value
    df = df.dropna(subset=["canonical_smiles", "standard_value"])
    print("After dropping NaNs:", df.shape)

    # Save
    df.to_csv(OUT_CSV, index=False)
    print(f"\n✅ Saved raw HIA/permeability data to: {OUT_CSV}")

    return df


if __name__ == "__main__":
    df = fetch_absorption_from_chembl()
    print(df.head())


C:\Users\kiran\anaconda3\envs\chemtox_env\lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


Fetching type: PCaco-2
  → got 1 records
Fetching type: Papp
  → got 17956 records
Fetching type: Papp e-6
  → got 8 records
Fetching type: Permeability
  → got 564 records
Fetching type: Permeability coefficient
  → got 37 records
Fetching type: Absorption
  → got 144 records
Fetching type: Pcaco2
  → got 14 records
Fetching type: Cell permeation
  → got 13 records
Fetching type: Papps
  → got 7 records
Fetching type: Permeability rate
  → got 4 records
Fetching type: permeability
  → got 15235 records
Fetching type: logPapp
  → got 916 records
Fetching type: LogPapp
  → got 38 records
Fetching type: Papp A to B (mean)
  → got 274 records
Fetching type: Caco-2 A-B
  → got 3 records
Fetching type: Caco-2 Papp
  → got 34 records
Fetching type: Caco-2 permeability
  → got 4 records
Fetching type: Drugabsorption
  → got 191 records

Total raw rows collected: 35443
Initial DataFrame shape: (35443, 9)
After dropping NaNs: (35437, 9)

✅ Saved raw HIA/permeability data to: chembl_hia_raw_from